# Lab 2C: Completions + Embeddings in Python

**Time**: ~30 min  
**Environment**: Jupyter kernel in VS Code  

In this exercise you will learn Azure OpenAI chat completions, streaming, and embeddings generation. Practice cosine similarity calculations with generated vectors.

The lab follows the same structure as the C# version. Run each cell in order to complete the steps.

In [ ]:
%pip install azure-cosmos azure-identity openai python-dotenv numpy --quiet

## Step 0: Initialize Connection

Set up Cosmos DB and Azure OpenAI client connections.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ.get("COSMOS_ENDPOINT")
FOUNDRY_ENDPOINT = os.environ.get("FOUNDRY_ENDPOINT")
EMBEDDINGS_ENDPOINT = os.environ.get("EMBEDDINGS_ENDPOINT")
EMBEDDINGS_KEY = os.environ.get("EMBEDDINGS_KEY")
DB_NAME = "WorkshopData"
COMPLETIONS_MODEL = os.environ.get("COMPLETIONS_MODEL", "phi-4-mini-instruct")
EMBEDDINGS_MODEL = os.environ.get("EMBEDDINGS_MODEL", "text-embedding-3-small")

for var in ["COSMOS_ENDPOINT", "FOUNDRY_ENDPOINT", "EMBEDDINGS_ENDPOINT", "EMBEDDINGS_KEY"]:
    if not os.environ.get(var):
        raise RuntimeError(f"{var} environment variable is required.")

print(f"Cosmos Endpoint:     {ENDPOINT}")
print(f"Foundry Endpoint:    {FOUNDRY_ENDPOINT}")
print(f"Embeddings Endpoint: {EMBEDDINGS_ENDPOINT}")
print(f"Database:            {DB_NAME}")
print(f"Completions Model:   {COMPLETIONS_MODEL}")
print(f"Embeddings Model:    {EMBEDDINGS_MODEL}")

In [ ]:
from azure.cosmos import CosmosClient
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

cred = DefaultAzureCredential()
cosmos_client = CosmosClient(url=ENDPOINT, credential=cred)
db = cosmos_client.get_database_client(DB_NAME)
print(f"Connected to: {ENDPOINT}/{DB_NAME}")

# Chat completions: Foundry endpoint, Entra ID auth.
token_provider = get_bearer_token_provider(cred, "https://ai.azure.com/.default")
foundry_client = OpenAI(
    base_url=f"{FOUNDRY_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)

# Embeddings: separate Azure OpenAI resource, API key auth
# (the v1 embeddings surface does not yet support Entra ID).
embeddings_client = OpenAI(
    base_url=f"{EMBEDDINGS_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=EMBEDDINGS_KEY,
)
print("Foundry chat client + embeddings client initialized")

## Step 1: Chat Completions

Replace the placeholder `completion_text` and `usage` in the code cell with a real chat completion call:

```python
completion = foundry_client.chat.completions.create(
    model=COMPLETIONS_MODEL,
    messages=messages,
    temperature=0.7,
    max_completion_tokens=200,
    top_p=0.95
)
completion_text = completion.choices[0].message.content
usage = completion.usage
```

**Expected output**: A response about Cosmos DB partitioning with token usage.

In [ ]:
messages = [
    {"role": "system", "content": "You are a data platform expert."},
    {"role": "user", "content": "Explain partitioning in Cosmos DB in 2 sentences."}
]

completion = foundry_client.chat.completions.create(
    model=COMPLETIONS_MODEL,
    messages=messages,
    temperature=0.7,
    max_completion_tokens=200,
    top_p=0.95
)
completion_text = completion.choices[0].message.content
usage = completion.usage

print(f"Response: {completion_text}")
print(f"Token usage - Prompt: {usage.prompt_tokens}, Completion: {usage.completion_tokens}")

## Step 2: Streaming Response

Replace the placeholder loop in the code cell with a real streaming call:

```python
for chunk in foundry_client.chat.completions.create(
    model=COMPLETIONS_MODEL,
    messages=messages,
    stream=True,
):
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
```

**Expected output**: Streaming text about Cosmos DB consistency levels.

In [ ]:
messages = [
    {"role": "user", "content": "List 5 Cosmos DB consistency levels and their use cases."}
]

print("Streaming response: ", end="")

for chunk in foundry_client.chat.completions.create(
    model=COMPLETIONS_MODEL,
    messages=messages,
    stream=True,
):
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

## Step 3: Generate Embeddings and Compare

The `texts` list and cosine similarity comparison are already in place. Replace the placeholder return in `get_embedding` with a real embeddings call:

```python
resp = embeddings_client.embeddings.create(input=text, model=EMBEDDINGS_MODEL)
return resp.data[0].embedding
```

**Expected output**: Embedding dimensions printed; cosine similarity between docs 1 and 3 noticeably higher than zero.

In [ ]:
import numpy as np


def get_embedding(text: str) -> list[float]:
    resp = embeddings_client.embeddings.create(input=text, model=EMBEDDINGS_MODEL)
    return resp.data[0].embedding


texts = [
    "Azure Cosmos DB is globally distributed.",
    "Microsoft Azure is a cloud platform.",
    "Cosmos DB vector search supports semantic similarity.",
]

embeddings = []
print("Generating embeddings:")
for text in texts:
    emb = get_embedding(text)
    embeddings.append((text, emb))
    print(f"  {text[:40]}... dim={len(emb)}")


def cosine_similarity(a: list[float], b: list[float]) -> float:
    a_arr = np.array(a)
    b_arr = np.array(b)
    denom = (np.linalg.norm(a_arr) * np.linalg.norm(b_arr))
    if denom == 0:
        return 0.0
    return float(np.dot(a_arr, b_arr) / denom)


similarity = cosine_similarity(embeddings[0][1], embeddings[2][1])
print()
print(f"Cosine similarity (docs 1 vs 3): {similarity:.4f}")
print("Note: Higher value = more similar (range: -1 to 1)")